# Notebook 17 — Mini-Project: Full Statistical Analysis on Public Expression Data

**Module:** 03 — Statistics and Probability  
**Notebook:** 17 of 18 — Portfolio artifact  
**Prerequisites:** NB01–NB16 (complete)  
**Time estimate:** 3–4 hours (spread over 2 sessions)

> **Track A + Track B portfolio artifact.** This notebook produces the Module 03
> portfolio figure saved to `portfolio/module03_statistical_analysis_report.png`.
> Every cell must be runnable from a clean environment. The dataset is fetched
> programmatically — never committed to the repository.

---
## 1. Project Statement

**Dataset:** GEO accession GSE2034 — breast cancer gene expression (Wang et al., 2005).
286 lymph-node-negative breast cancer patients; outcome: distant relapse within 5 years.

**Task:** Identify differentially expressed genes between relapse and no-relapse groups
using the statistical methods developed in Module 03. Produce a reproducible report
with pre-specified analysis plan (registered analysis, not HARKing).

**Pre-specified hypotheses (write these BEFORE looking at the data):**
1. There exist differentially expressed genes between relapse/no-relapse groups (FDR 5%).
2. Effect sizes will be heterogeneous — some genes have strong signal, most have none.
3. BH correction will discover substantially more genes than Bonferroni.

**Analysis plan:**
1. Data download and quality assessment
2. Descriptive statistics and sample clustering
3. Differential expression: Welch t-test + BH FDR correction
4. Visualization: volcano plot, MA plot, expression heatmap
5. Interpretation and limitations

In [ ]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

In [ ]:
# Cell 2 — Data acquisition
# GEO data for GSE2034 should be placed in ../datasets/raw/GSE2034/
# Run this cell to fetch via GEOparse if the data is not present.
#
# Instructions for manual download:
#   1. Visit: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE2034
#   2. Download: GSE2034_series_matrix.txt.gz
#   3. Place in:  curriculum/03_statistics_and_probability/datasets/raw/GSE2034/
#
# If GEOparse is installed:
#   import GEOparse
#   gse = GEOparse.get_GEO(geo='GSE2034', destdir='../datasets/raw/')

DATA_DIR = Path('../datasets/raw/GSE2034')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR.resolve())
print("Files present:", list(DATA_DIR.iterdir()) if DATA_DIR.exists() else "(empty — download required)")
print()
print("To proceed without the real dataset, run Cell 3 (synthetic fallback).")

In [ ]:
# Cell 3 — Synthetic fallback (runs the full analysis pipeline on simulated data)
# This cell produces a complete, reproducible analysis even without the GEO download.
# Replace with real data loading once GSE2034 is fetched.

rng = np.random.default_rng(2034)  # seed = accession for reproducibility

N_GENES   = 22_283   # Affymetrix HG-U133A array gene count
N_RELAPSE = 106      # cases with distant relapse
N_NOREC   = 180      # no distant relapse
N_DE_TRUE = 800      # true DE genes (for simulation)
N_SAMPLES = N_RELAPSE + N_NOREC

# Simulate microarray log2-expression (already normalized Affymetrix data)
base_expr = rng.normal(8.0, 2.5, N_GENES)      # gene-specific mean

de_genes = rng.choice(N_GENES, N_DE_TRUE, replace=False)
true_lfc = np.zeros(N_GENES)
true_lfc[de_genes] = rng.choice([-1,1], N_DE_TRUE) * rng.uniform(0.3, 1.5, N_DE_TRUE)

# Expression matrix: genes × samples
X_ctrl = rng.normal(base_expr[:, None], 0.8, (N_GENES, N_NOREC))
X_case = rng.normal((base_expr + true_lfc)[:, None], 0.8, (N_GENES, N_RELAPSE))
expr_matrix = np.hstack([X_ctrl, X_case])       # (genes, samples)

labels = np.array(['norelapse'] * N_NOREC + ['relapse'] * N_RELAPSE)
is_relapse = labels == 'relapse'

print(f"Expression matrix: {N_GENES:,} genes × {N_SAMPLES} samples")
print(f"Groups: {N_RELAPSE} relapse, {N_NOREC} no-relapse")
print(f"True DE genes: {N_DE_TRUE:,} ({N_DE_TRUE/N_GENES:.1%})")

In [ ]:
# Cell 4 — Quality assessment: per-sample mean and variance
sample_means = expr_matrix.mean(axis=0)
sample_sds   = expr_matrix.std(axis=0)

print("Per-sample expression summary:")
print(f"  Global mean: {sample_means.mean():.3f} ± {sample_means.std():.3f}")
print(f"  Min mean: {sample_means.min():.3f}, Max mean: {sample_means.max():.3f}")

# Flag outlier samples (>3 SD from mean of means)
mean_of_means = sample_means.mean()
sd_of_means   = sample_means.std()
outliers = np.where(np.abs(sample_means - mean_of_means) > 3 * sd_of_means)[0]
print(f"  Outlier samples (>3 SD): {len(outliers)} samples")

In [ ]:
# Cell 5 — Differential expression analysis
ctrl_expr = expr_matrix[:, ~is_relapse]
case_expr = expr_matrix[:, is_relapse]

log2fc = case_expr.mean(axis=1) - ctrl_expr.mean(axis=1)
p_vals = np.array([
    stats.ttest_ind(ctrl_expr[g], case_expr[g], equal_var=False)[1]
    for g in range(N_GENES)
])

# BH FDR correction
_, q_bh, _, _  = multipletests(p_vals, method='fdr_bh')
# Bonferroni
q_bonf = np.minimum(p_vals * N_GENES, 1.0)

alpha = 0.05
sig_bh   = q_bh   < alpha
sig_bonf = q_bonf < alpha

print(f"Differential expression results (FDR {alpha*100:.0f}%):")
print(f"  BH significant genes:         {sig_bh.sum():,}")
print(f"  Bonferroni significant genes:  {sig_bonf.sum():,}")
print(f"  Up-regulated (BH, lfc>0):     {(sig_bh & (log2fc > 0)).sum():,}")
print(f"  Down-regulated (BH, lfc<0):   {(sig_bh & (log2fc < 0)).sum():,}")

# Evaluate against ground truth (simulation only)
de_truth = np.zeros(N_GENES, dtype=bool)
de_truth[de_genes] = True
tp = (sig_bh & de_truth).sum()
fp = (sig_bh & ~de_truth).sum()
print(f"\n  True positives (simulation): {tp:,}")
print(f"  False positives (simulation): {fp:,}")
print(f"  Sensitivity: {tp/N_DE_TRUE:.1%}")
print(f"  Empirical FDR: {fp/max(sig_bh.sum(),1):.1%}")

In [ ]:
# Cell 6 — Portfolio figure: 4-panel statistical analysis report
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.40, wspace=0.35)

# --- Panel A: Volcano plot ---
ax = fig.add_subplot(gs[0, 0])
neg_logq = -np.log10(q_bh + 1e-300)
ax.scatter(log2fc[~sig_bh], neg_logq[~sig_bh], s=2, alpha=0.2, color='gray')
up_tp = sig_bh & de_truth & (log2fc > 0)
dn_tp = sig_bh & de_truth & (log2fc < 0)
fp_m  = sig_bh & ~de_truth
ax.scatter(log2fc[up_tp], neg_logq[up_tp], s=8, alpha=0.7, color='tomato',    label=f'Up DE (n={up_tp.sum()})')
ax.scatter(log2fc[dn_tp], neg_logq[dn_tp], s=8, alpha=0.7, color='steelblue', label=f'Down DE (n={dn_tp.sum()})')
ax.scatter(log2fc[fp_m],  neg_logq[fp_m],  s=8, alpha=0.7, color='orange',    label=f'FP (n={fp_m.sum()})')
ax.axhline(-np.log10(alpha), color='black', ls='--', lw=0.8)
ax.set_xlabel('log₂FC (relapse vs. no-relapse)', fontsize=10)
ax.set_ylabel('-log₁₀(q)', fontsize=10)
ax.set_title('A. Volcano plot — BH FDR 5%', fontsize=11)
ax.legend(fontsize=8)

# --- Panel B: MA plot ---
ax = fig.add_subplot(gs[0, 1])
mean_expr = expr_matrix.mean(axis=1)
ax.scatter(mean_expr[~sig_bh], log2fc[~sig_bh], s=2, alpha=0.15, color='gray')
ax.scatter(mean_expr[sig_bh & de_truth], log2fc[sig_bh & de_truth],
           s=8, alpha=0.6, color='steelblue', label='True DE')
ax.scatter(mean_expr[fp_m], log2fc[fp_m],
           s=8, alpha=0.6, color='orange', label='False positive')
ax.axhline(0, color='black', lw=0.5, ls='--')
ax.set_xlabel('Mean log₂ expression', fontsize=10)
ax.set_ylabel('log₂FC', fontsize=10)
ax.set_title('B. MA plot', fontsize=11)
ax.legend(fontsize=8)

# --- Panel C: p-value distribution ---
ax = fig.add_subplot(gs[1, 0])
ax.hist(p_vals, bins=50, color='steelblue', edgecolor='none', density=True, alpha=0.85)
ax.axhline(1.0, color='gray', ls='--', lw=1, label='Uniform null')
ax.set_xlabel('Raw p-value', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('C. p-value histogram', fontsize=11)
ax.legend(fontsize=8)

# --- Panel D: Heatmap top 50 DE genes ---
ax = fig.add_subplot(gs[1, 1])
top50_idx = np.argsort(q_bh)[:50]
H = expr_matrix[top50_idx]
H_z = (H - H.mean(axis=1, keepdims=True)) / (H.std(axis=1, keepdims=True) + 1e-9)
# Sort samples: no-relapse then relapse
order = np.concatenate([np.where(~is_relapse)[0], np.where(is_relapse)[0]])
H_z = H_z[:, order]
im = ax.imshow(H_z, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2, interpolation='none')
ax.axvline(N_NOREC - 0.5, color='yellow', lw=1.5)
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('Samples (no-relapse | relapse)', fontsize=10)
ax.set_ylabel('Top 50 DE genes', fontsize=10)
ax.set_title('D. Expression heatmap (z-scored)', fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03, label='Z-score')

fig.suptitle(
    'Module 03 — Statistical Analysis Report\n'
    f'Breast cancer relapse DE analysis (GSE2034-equivalent, n={N_SAMPLES})\n'
    f'BH FDR 5%: {sig_bh.sum():,} DE genes | Sensitivity={tp/N_DE_TRUE:.1%} | FDR={fp/max(sig_bh.sum(),1):.1%}',
    fontsize=12, fontweight='bold'
)

portfolio_path = Path('../../../portfolio/module03_statistical_analysis_report.png')
portfolio_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(portfolio_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Portfolio figure saved → {portfolio_path.resolve()}")

---
## Interpretation and Limitations

**Findings:**
- The analysis identified _____ differentially expressed genes at BH FDR 5%.
- BH correction found substantially more genes than Bonferroni, consistent with
  hypothesis 3 (pre-specified).
- The p-value histogram shows an anti-conservative spike near 0, consistent with
  real signal in the data (hypothesis 1 supported).

**Limitations:**
1. The t-test assumes approximate normality of log2-expression within groups.
   For microarray data this is generally reasonable; for raw RNA-seq counts it
   would not be — a negative binomial model would be required.
2. No covariate adjustment (batch effects, age, ER status). Real GSE2034 analysis
   should include clinical covariates.
3. This analysis did not perform gene set enrichment — the biological interpretation
   of individual gene lists is limited without pathway context.

> *[Fill in findings with actual numbers after running the notebook.]*

---
## Reflection

**Date completed:** ____________________

> *[Did the pre-specified hypotheses hold? What surprised you? What would you do differently
if you were reporting this as a real analysis?]*

---
*Next: `18_module_assessment_mock_interview.ipynb`*